In [2]:
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project


In [3]:
from python.src.train.model_trainer import train_model
from pathlib import Path
import torch
import os

PROJECT_ROOT = Path.cwd()

models = ["asymmetric_trinet"]
version = "v2"
seeds = [216]
n_per_class = [2500]
epochs = 35


for m in models:
  for s in seeds:
    for n in n_per_class:

      print(f"\n\nRunning experiment model = {m}, seed = {s}, n_per_class = {n}")
      print("============================================================================\n")

      trained_model = train_model(seed=s,
                  project_root=PROJECT_ROOT,
                  model_name=m,
                  n_per_class=n,
                  spec_version=version,
                  n_epochs=epochs
                  )
      # clean up RAM
      del trained_model
      if torch.cuda.is_available():
            torch.cuda.empty_cache()
      elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
      print("\nTraining finished")



Running experiment model = asymmetric_trinet, seed = 216, n_per_class = 2500


Validation report found: /content/drive/Othercomputers/My Laptop/thesis_project/reports/validations/validation_seed216_n2500_v2.json
Starting training...

Using device: cuda

Epoch 01 | LR: 0.001000 | Train: 2.1772 (CE 1.8680 / SupCon 3.0922) | Val Loss: 1.3547 | Val Acc: 70.68
Epoch 05 | LR: 0.000968 | Train: 1.2446 (CE 1.0336 / SupCon 2.1093) | Val Loss: 0.9437 | Val Acc: 82.62
Epoch 10 | LR: 0.000846 | Train: 1.1324 (CE 0.9365 / SupCon 1.9586) | Val Loss: 0.8327 | Val Acc: 87.02
Epoch 15 | LR: 0.000655 | Train: 1.0506 (CE 0.8667 / SupCon 1.8390) | Val Loss: 0.7952 | Val Acc: 88.36
Epoch 20 | LR: 0.000433 | Train: 0.9496 (CE 0.7825 / SupCon 1.6711) | Val Loss: 0.8002 | Val Acc: 89.34
Epoch 25 | LR: 0.000225 | Train: 0.8861 (CE 0.7304 / SupCon 1.5571) | Val Loss: 0.8060 | Val Acc: 89.82
Epoch 30 | LR: 0.000071 | Train: 0.8203 (CE 0.6770 / SupCon 1.4328) | Val Loss: 0.8386 | Val Acc: 89.46
Epoch 35 | LR: 0

In [3]:
from python.src.train.osr_trainer import train_osr_model
from python.src.train.osr_hparams import OSRHParams
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()

version = "v2"
seeds = [216]
n_per_class = [2500]
epochs = 30

# Initialize hyperparameters (tweak these directly here if needed)
hparams = OSRHParams()

for s in seeds:
    for n in n_per_class:

        print(f"\n\nRunning OSR experiment | seed = {s}, n_per_class = {n}")
        print("============================================================================\n")

        trained_model = train_osr_model(
            seed=s,
            n_per_class=n,
            spec_version=version,
            project_root=PROJECT_ROOT,
            epochs=epochs,
            hparams=hparams
        )

        # clean up RAM
        del trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()

        print("Trainng Complete")



Running OSR experiment | seed = 216, n_per_class = 2500


OsrSAF_TriNet | seed=216 | n=2500
Device              : cuda
Closed-set ckpt     : asymmetric_trinet_seed216_n2500.pt
Codebook fill epochs: 15
Phase 2 (calibrator): until epoch 30
FPR cap (Youden)    : 0.40
Recal interval      : 5

[load_osr_datasets] proxy unknowns: 7000 samples (train 5600 / val 1400)
[load_osr_datasets] test  unknowns: 3000 samples (held out)
[OsrSAF_TriNet] Loaded backbone from /content/drive/Othercomputers/My Laptop/thesis_project/artifacts/checkpoints/asymmetric_trinet_seed216_n2500.pt

[Stage 2.A] Populating codebook over 15 epochs (frozen backbone)

  Fill epoch 1/15 | init=100% | spread=0.0785 | updates/centroid=381.7
  Fill epoch 2/15 | init=100% | spread=0.0824 | updates/centroid=761.2
  Fill epoch 3/15 | init=100% | spread=0.0915 | updates/centroid=1140.1
  Fill epoch 4/15 | init=100% | spread=0.0874 | updates/centroid=1519.9
  Fill epoch 5/15 | init=100% | spread=0.0892 | updates/centroid=1894.9
 

In [ ]:
import torch
import platform
import psutil

# 1. GPU Info
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# 2. CPU Info
cpu_info = platform.processor()

# 3. RAM Info
ram_total = psutil.virtual_memory().total / (1024**3)

# 4. PyTorch Version
torch_ver = torch.__version__

print(f"--- THESIS HARDWARE SPECS ---")
print(f"GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
print(f"CPU: {cpu_info} (Intel Xeon)")
print(f"System RAM: {ram_total:.2f} GB")
print(f"PyTorch Version: {torch_ver}")
print(f"-----------------------------")


In [ ]:
# ============================================================
#  BASELINE TRAINING CELL — paste into existing Colab notebook
#  Trains VGG-16, ResNet-18, DenseNet-121 with two-phase
#  fine-tuning. Already-trained checkpoints are skipped.
# ============================================================

from google.colab import drive
from pathlib import Path
import sys, os, json, random
from datetime import datetime, timezone

import torch
import torch.nn as nn
import numpy as np

# ── 1. Mount Drive and set project root ──────────────────────
drive.mount('/content/drive', force_remount=True)   # force_remount=False skips
                                                      # re-mount if already mounted

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

# ── 2. Imports (after path is set) ───────────────────────────
from python.src.legacy_models import (
    #LiteratureBaseline_VGG16,
    LiteratureBaseline_ResNet18,
    #LiteratureBaseline_DenseNet121,
)
from python.src.dataio import load_artifact
from python.src.preprocessing import build_feature_tensor, split_dataset
from python.src.utils import (
    create_train_loader, create_eval_loader,
    resolve_device, FeatureTensorDataset,
)

# ── 3. Config — edit here if needed ──────────────────────────
SEED          = 55
N_PER_CLASS   = 2500
SPEC_VER      = "v2"
N_EPOCHS      = 50
PHASE1_EPOCHS = 15       # head-only warm-up epochs
LR_PHASE1     = 1e-3     # higher LR for randomly-init'd head
LR_PHASE2     = 1e-4     # lower LR for pretrained backbone layers
WEIGHT_DECAY  = 5e-3
BATCH_SIZE    = 32
NUM_CLASSES   = 10

BASELINE_REGISTRY = {
    #"vgg_16":       LiteratureBaseline_VGG16,
    "resnet_18":    LiteratureBaseline_ResNet18,
    #"densenet_121": LiteratureBaseline_DenseNet121,
}

# ── 4. Helpers ────────────────────────────────────────────────
def _set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

def _make_loader(split_tuple, batch_size, shuffle):
    x_stft, x_iq, x_if, y = split_tuple
    ds = FeatureTensorDataset(x_stft, x_iq, x_if, y)
    return create_train_loader(ds, batch_size) if shuffle else create_eval_loader(ds, batch_size)

@torch.no_grad()
def _evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for x_stft, x_iq, x_if, y in loader:
        x_stft, x_iq, x_if, y = x_stft.to(device), x_iq.to(device), x_if.to(device), y.to(device)
        logits = model(x_stft, x_iq, x_if)
        total_loss    += criterion(logits, y).item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n       += y.size(0)
    return total_loss / total_n, total_correct / total_n

def _run_phase(model, train_loader, val_loader, optimizer, scheduler,
               criterion, device, start_ep, end_ep, phase_label, log_epochs):
    best_acc, best_state = 0.0, None
    for epoch in range(start_ep, end_ep + 1):
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x_stft, x_iq, x_if, y in train_loader:
            x_stft, x_iq, x_if, y = x_stft.to(device), x_iq.to(device), x_if.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_stft, x_iq, x_if)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()
            bs = y.size(0)
            total_loss    += loss.item() * bs
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n       += bs

        train_loss = total_loss / total_n
        train_acc  = total_correct / total_n
        val_loss, val_acc = _evaluate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch == start_ep or epoch % 5 == 0 or epoch == end_ep:
            print(f"    [{phase_label}] Ep {epoch:02d}/{end_ep} | "
                  f"LR {lr_now:.2e} | Train {train_loss:.4f} ({100*train_acc:.1f}%) | "
                  f"Val {val_loss:.4f} ({100*val_acc:.2f}%)")

        log_epochs.append({"epoch": epoch, "phase": phase_label,
                           "train_loss": train_loss, "train_acc": train_acc,
                           "val_loss": val_loss, "val_accuracy": val_acc, "lr": lr_now})
    return best_acc, best_state

# ── 5. Load dataset once (shared across all three models) ────
print("\nLoading dataset...")

_train_path = (PROJECT_ROOT / "artifacts" / "datasets" / "impaired"
               / f"impaired_dataset_{SPEC_VER}_seed{SEED}_n{N_PER_CLASS}_train.mat")
_eval_path  = (PROJECT_ROOT / "artifacts" / "datasets" / "impaired"
               / f"impaired_dataset_{SPEC_VER}_seed{SEED}_n{N_PER_CLASS}_eval.mat")

_train_artifact = load_artifact(str(_train_path), load_params=False)
_x_stft, _x_iq, _x_if, _y = build_feature_tensor(_train_artifact)

# split_dataset returns two tuples — kept as tuples throughout
train_set, val_set = split_dataset(_x_stft, _x_iq, _x_if, _y, train_ratio=0.8, seed=SEED)

# Extract tensors from the Subsets (as you already know works)
x_stft_tr = train_set.dataset.tensors[0][train_set.indices]
x_iq_tr = train_set.dataset.tensors[1][train_set.indices]
x_if_tr = train_set.dataset.tensors[2][train_set.indices]
y_tr = train_set.dataset.tensors[3][train_set.indices]
x_stft_val = val_set.dataset.tensors[0][val_set.indices]
x_iq_val = val_set.dataset.tensors[1][val_set.indices]
x_if_val = val_set.dataset.tensors[2][val_set.indices]
y_val = val_set.dataset.tensors[3][val_set.indices]

train_split = (x_stft_tr, x_iq_tr, x_if_tr, y_tr)
val_split = (x_stft_val, x_iq_val, x_if_val, y_val)

_eval_artifact = load_artifact(str(_eval_path), load_params=False)
_xs, _xi, _xf, _yt = build_feature_tensor(_eval_artifact)
test_data = (_xs, _xi, _xf, _yt)

print(f"  Train : {train_split[0].shape[0]} | Val : {val_split[0].shape[0]} | Test : {test_data[0].shape[0]}")

device = resolve_device("auto")
print(f"  Device: {device}")

# ── 6. Train all baselines ────────────────────────────────────
for model_name, model_cls in BASELINE_REGISTRY.items():

    ckpt_path = (PROJECT_ROOT / "artifacts" / "checkpoints"
                 / f"{model_name}_baseline_seed{SEED}_n{N_PER_CLASS}.pt")

    if ckpt_path.exists():
        print(f"\n[SKIP] {model_name} — checkpoint already exists.")
        continue

    print(f"\n{'='*65}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'='*65}")

    _set_seed(SEED)
    model     = model_cls(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    train_loader = _make_loader(train_split, BATCH_SIZE, shuffle=True)
    val_loader   = _make_loader(val_split,   BATCH_SIZE, shuffle=False)
    test_loader  = _make_loader(test_data,   BATCH_SIZE, shuffle=False)

    log_epochs      = []
    global_best_acc = 0.0
    global_best_state = None

    # Phase 1 — head only
    model.freeze_for_phase1()
    p1_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1 trainable params : {p1_params:,}")

    opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                             lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

    best1, state1 = _run_phase(model, train_loader, val_loader, opt1, sch1,
                                criterion, device, 1, PHASE1_EPOCHS,
                                "Phase1", log_epochs)
    if best1 > global_best_acc:
        global_best_acc, global_best_state = best1, state1

    # Phase 2 — last block + head
    model.unfreeze_for_phase2()
    p2_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 2 trainable params : {p2_params:,}")

    phase2_len = N_EPOCHS - PHASE1_EPOCHS
    opt2 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                             lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=phase2_len)

    best2, state2 = _run_phase(model, train_loader, val_loader, opt2, sch2,
                                criterion, device,
                                PHASE1_EPOCHS + 1, N_EPOCHS,
                                "Phase2", log_epochs)
    if best2 > global_best_acc:
        global_best_acc, global_best_state = best2, state2

    # Final test
    model.load_state_dict(global_best_state)
    test_loss, test_acc = _evaluate(model, test_loader, criterion, device)
    print(f"\n  Best Val Acc : {100*global_best_acc:.2f}%")
    print(f"  Test Acc     : {100*test_acc:.2f}%")

    # Save checkpoint
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(global_best_state, ckpt_path)
    print(f"  Checkpoint   : {ckpt_path.name}")

    # Save log
    log_dir = PROJECT_ROOT / "artifacts" / "logs" / "baselines"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_baseline_seed{SEED}_n{N_PER_CLASS}.json"
    with log_path.open("w") as f:
        json.dump({"model": model_name, "seed": SEED, "n_per_class": N_PER_CLASS,
                   "best_val_acc": global_best_acc, "test_acc": test_acc,
                   "test_loss": test_loss, "epochs": log_epochs}, f, indent=2)
    print(f"  Log          : {log_path.name}")

    del model
    torch.cuda.empty_cache()

print("\n\n✓ All baselines complete.")